In [1]:
import io
import zipfile
import requests
import frontmatter

In [3]:
import requests
resp = requests.get('https://httpbin.org/get', timeout=10)
print(resp.status_code)

200


In [4]:
url = 'https://github.com/DataTalksClub/faq/archive/refs/heads/main.zip'
resp = requests.get(url, timeout=30)
print(resp.status_code)

200


In [5]:
repository_data = []

zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename
    filename_lower = filename.lower()

    if not (filename_lower.endswith('.md') or filename_lower.endswith('.mdx')):
        continue

    try:
        with zf.open(file_info) as f_in:
            content = f_in.read().decode('utf-8', errors='ignore')
            post = frontmatter.loads(content)
            data = post.to_dict()
            data['filename'] = filename
            repository_data.append(data)
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        continue

zf.close()
print(f"Found {len(repository_data)} documents")

Found 1285 documents


In [9]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [10]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
print(dtc_faq[0])

{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -r

In [8]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

Found 95 documents


In [11]:
zoomcamp = read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')
print(f"Found {len(zoomcamp)} documents")

Found 143 documents
